# 全连接网络 MLP：深度学习的第一步
> 难度标签：高级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：06_深度学习 | 源文件：全连接网络_MLP.py | 核心功能：PyTorch 全连接层、前向传播、训练循环

## 概述

多层感知机（Multi-Layer Perceptron, MLP）是最基础的深度学习模型。它由多个全连接层组成，每层的每个神经元与上一层所有神经元相连。虽然简单，但 MLP 是理解深度学习的起点——CNN、RNN、Transformer 都是在此基础上的改进。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_digits
# --- 导入库 ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## 数学原理

### 1. 前向传播

**代码对应**：MLP 由多层全连接层组成。

单层前向传播：$\mathbf{h} = \sigma(\mathbf{W}\mathbf{x} + \mathbf{b})$

多层（$L$ 层）：
$$\mathbf{h}^{(0)} = \mathbf{x}, \quad \mathbf{h}^{(l)} = \sigma(\mathbf{W}^{(l)}\mathbf{h}^{(l-1)} + \mathbf{b}^{(l)}), \quad \hat{y} = \mathbf{h}^{(L)}$$

常用激活函数：
- ReLU：$\sigma(z) = \max(0, z)$
- Sigmoid：$\sigma(z) = 1/(1+e^{-z})$
- Tanh：$\sigma(z) = (e^z - e^{-z})/(e^z + e^{-z})$

### 2. 反向传播（Backpropagation）

**链式法则**：$\frac{\partial L}{\partial \mathbf{W}^{(l)}} = \frac{\partial L}{\partial \mathbf{h}^{(l)}} \cdot \frac{\partial \mathbf{h}^{(l)}}{\partial \mathbf{W}^{(l)}}$

误差从输出层逐层向前传播：
$$\boldsymbol{\delta}^{(L)} = \nabla_{\hat{y}}L \odot \sigma'(\mathbf{z}^{(L)})$$
$$\boldsymbol{\delta}^{(l)} = (\mathbf{W}^{(l+1)T}\boldsymbol{\delta}^{(l+1)}) \odot \sigma'(\mathbf{z}^{(l)})$$

### 3. 梯度下降优化

**SGD**：$\theta \leftarrow \theta - \alpha\nabla_\theta L$

**Adam**：结合动量和自适应学习率：
$$m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t, \quad v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2$$
$$\hat{m}_t = m_t/(1-\beta_1^t), \quad \hat{v}_t = v_t/(1-\beta_2^t)$$
$$\theta \leftarrow \theta - \alpha\frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

### 4. 梯度消失与梯度爆炸

梯度在反向传播中逐层相乘：$\frac{\partial L}{\partial \mathbf{W}^{(1)}} \propto \prod_{l=1}^{L-1}\mathbf{W}^{(l)}\text{diag}(\sigma'(\mathbf{z}^{(l)}))$

- Sigmoid/Tanh 的导数 $\leq 0.25$，连乘后梯度指数衰减（梯度消失）
- ReLU 的导数为 0 或 1，有效缓解梯度消失
- Batch Normalization 也帮助稳定梯度

### 1. 加载数据

首先加载数据集，为后续训练和评估做准备。

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 标准化
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 转为 Tensor
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.LongTensor(y_train)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.LongTensor(y_test)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

### 2. 定义 MLP 模型

运行 2. 定义 MLP 模型 的代码，观察算法在该环节的行为。

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim=64, hidden_dims=[128, 64], n_classes=10, dropout=0.2):
        super().__init__()
        layers = []
        prev_dim = input_dim
# --- 循环处理 ---
        for h_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = h_dim
# --- 继续 ---
        layers.append(nn.Linear(prev_dim, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model = MLP()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"=== MLP 模型结构 ===")
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"总参数量: {total_params:,}")

### 3. 训练

执行模型训练循环，观察损失函数的收敛过程。

In [ ]:
print("\n=== 训练 ===")
n_epochs = 30
for epoch in range(n_epochs):
    model.train()
    total_loss = 0
# --- 赋值/配置 ---
    correct = 0
    total = 0
    for batch_X, batch_y in train_loader:
        output = model(batch_X)
        loss = criterion(output, batch_y)
# --- 继续 ---
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (output.argmax(dim=1) == batch_y).sum().item()
# --- 继续 ---
        total += batch_y.size(0)

    if (epoch + 1) % 5 == 0:
        train_acc = correct / total
        model.eval()
        with torch.no_grad():
            test_output = model(X_test_t)
# --- 继续 ---
            test_acc = (test_output.argmax(dim=1) == y_test_t).float().mean().item()
        print(f"  Epoch {epoch+1:>2}: Loss={total_loss/len(train_loader):.4f}, "
              f"Train Acc={train_acc:.4f}, Test Acc={test_acc:.4f}")

### 4. 不同网络结构对比

对比不同模型或配置的性能差异。

In [ ]:
print("\n=== 不同网络结构对比 ===")
configs = {
    "浅 (64→10)": [64, 10],
    "中 (64→128→10)": [64, 128, 10],
    "深 (64→128→64→10)": [64, 128, 64, 10],
# --- 继续 ---
    "宽 (64→256→128→10)": [64, 256, 128, 10],
}
for name, hidden in configs.items():
    m = MLP(hidden_dims=hidden[1:-1] if len(hidden) > 2 else [], n_classes=10)
    opt = optim.Adam(m.parameters(), lr=0.001)
# --- 循环处理 ---
    for _ in range(20):
        m.train()
        for bx, by in train_loader:
            out = m(bx)
            loss = criterion(out, by)
# --- 继续 ---
            opt.zero_grad()
            loss.backward()
            opt.step()
    m.eval()
    with torch.no_grad():
# --- 继续 ---
        acc = (m(X_test_t).argmax(1) == y_test_t).float().mean().item()
    n_params = sum(p.numel() for p in m.parameters())
    print(f"  {name:>25}: Test Acc={acc:.4f}, 参数量={n_params:,}")

### 5. 常用技巧

运行 5. 常用技巧 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 深度学习常用技巧 ===")
print("1. Dropout: 随机丢弃神经元，防止过拟合")
print("2. BatchNorm: 稳定训练，加速收敛")
print("3. 学习率调度: 随训练逐步降低学习率")
print("4. 权重初始化: Xavier/He 初始化")
# --- 输出结果 ---
print("5. 早停 (Early Stopping): 验证集性能不再提升时停止")

## 关键代码解释

### 典型 MLP 结构

```python
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 10)
        )
```

每层结构：线性变换 + 非线性激活。没有激活函数，多层线性变换等价于单层。

### 训练循环

```python
for epoch in range(epochs):
    output = model(X)
    loss = criterion(output, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
```

## 注意事项

1. **激活函数必须有**：否则多层等价于单层
2. **梯度消失**：Sigmoid/Tanh 在深层网络中梯度消失，用 ReLU
3. **权重初始化**：不当初始化会导致训练失败

## 总结与延伸

以上代码展示了 **全连接网络 MLP** 的完整流程。

**解读要点**：
- 关注 **损失函数** 的收敛曲线
- 观察训练/验证损失是否出现分叉（过拟合信号）
- 对比不同超参数（学习率、层数）的影响

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **通用近似定理**：单隐层 MLP 可以逼近任意连续函数
- **Batch Normalization**：加速训练、稳定梯度
- **Dropout**：正则化手段，随机丢弃神经元
